# Dimensionality Reduction for Return Prediction, and Clustering Value/Size Portfolios

**UCLA MFE 413 — Machine Learning in Finance, Homework 6** (individual submission)

## Part 1: PCA vs. raw factors for predicting the S&P 500

Six stocks (Google, Walmart, Pfizer, Intel, GM, Apple) span very different sectors and risk
profiles. The question: does compressing their daily returns into two principal-component
portfolios lose predictive power for the S&P 500, versus regressing on all six stocks directly?

**Setup.** Fit PCA on daily returns during a training window (Apr–Aug 2021), build two
PCA-portfolio return series from the first two components, then compare an out-of-sample
(Aug–Oct 2021) regression of S&P 500 returns on the two PCA portfolios against a regression
on all six raw stock returns.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

df = pd.read_csv("Problem3Data.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# Price columns
stock_cols = [
    "close_google",
    "close_walmart",
    "close_pfe",
    "close_intc",
    "close_gm",
    "close_aapl"
]

# daily returns
returns = df[["Date", "price_sp"] + stock_cols].copy()
returns["sp_ret"] = returns["price_sp"].pct_change()

for col in stock_cols:
    returns[col + "_ret"] = returns[col].pct_change()

returns = returns.dropna().reset_index(drop=True)

stock_ret_cols = [col + "_ret" for col in stock_cols]

# Training and test periods
train = returns[(returns["Date"] >= "2021-04-01") & (returns["Date"] <= "2021-08-01")]
test = returns[(returns["Date"] > "2021-08-01") & (returns["Date"] <= "2021-10-01")]

X_train = train[stock_ret_cols]
X_test = test[stock_ret_cols]

y_train = train["sp_ret"]
y_test = test["sp_ret"]

### (a) First two principal components

The loadings on the first component (`w1`) are dominated by GM (0.73) and Intel (0.44) —
the two more volatile, cyclical names — while the second component (`w2`) contrasts GM
against Walmart and Intel, picking up a different axis of co-movement. Together the first
two components explain about 68% of total variance across the six stocks
(45.1% + 23.3%).

In [ ]:
# 3(a)
pca = PCA(n_components=2)
pca.fit(X_train)

w1 = pca.components_[0]
w2 = pca.components_[1]

print("Order of stocks:", stock_cols)
print("\nFirst principal component w1:", w1)
print("Second principal component w2:", w2)
print("\nExplained variance ratios:", pca.explained_variance_ratio_)

**Result:** `w1 = [0.342, 0.148, 0.025, 0.445, 0.731, 0.357]`,
`w2 = [-0.389, -0.085, -0.118, -0.428, 0.675, -0.434]`, explained variance ratios
`[0.451, 0.233]`.

### (b) Build the two PCA-portfolio return series

$x_{1,t} = w_1' r_t$ and $x_{2,t} = w_2' r_t$ — project each day's six-stock return vector onto
the two principal directions.

In [ ]:
# 3(b)
# x1_t = w1' r_t
# x2_t = w2' r_t

train_pca = pca.transform(X_train)
test_pca = pca.transform(X_test)

train["x1"] = train_pca[:, 0]
train["x2"] = train_pca[:, 1]

test["x1"] = test_pca[:, 0]
test["x2"] = test_pca[:, 1]

print(test[["Date", "x1", "x2"]].head())

### (c) Out-of-sample comparison: 2 PCA portfolios vs. 6 raw stocks

In [ ]:
# 3(c)
model_pca = LinearRegression()
model_pca.fit(train[["x1", "x2"]], y_train)

y_pred_pca = model_pca.predict(test[["x1", "x2"]])
r2_pca = r2_score(y_test, y_pred_pca)

print("Regression coefficients using PCA portfolios:", model_pca.coef_)
print("Intercept:", model_pca.intercept_)
print("Test R-squared using PCA portfolios:", r2_pca)

# Compared to direct regression on the 6 stock returns
model_direct = LinearRegression()
model_direct.fit(X_train, y_train)

y_pred_direct = model_direct.predict(X_test)
r2_direct = r2_score(y_test, y_pred_direct)

print("\nDirect regression on the original 6 stocks")
print("Regression coefficients:", model_direct.coef_)
print("Intercept:", model_direct.intercept_)
print("Test R-squared:", r2_direct)

print("\nComparison")
print("R-squared using PCA portfolios:", r2_pca)
print("R-squared using 6 stocks directly:", r2_direct)

**Result:** the 2-portfolio PCA regression achieves test R² = 0.767, versus 0.771 for the
full 6-stock regression — using two-thirds fewer predictors costs essentially nothing in
out-of-sample fit (a 0.4-point R² gap). Compressing six correlated return series down to two
orthogonal risk factors preserves almost all of the predictive signal, which is the point of
PCA as a dimensionality-reduction step before regression: fewer parameters to overfit, for
a negligible loss in explanatory power.

## Part 2: Clustering value/size portfolios by return pattern

Rather than clustering on static characteristics, cluster the 25 Fama-French size/book-to-market
portfolios directly on their *return time series* — five months (Sep 2020–Jan 2021) treated as
features — to see which portfolios moved together through that window, independent of their
size/value labels.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

ff = pd.read_csv("Fama_French_25_Portfolios.csv", skiprows=15)
ff = ff.rename(columns={ff.columns[0]: "Date"})

ff = ff[pd.to_numeric(ff["Date"], errors="coerce").notna()]
ff["Date"] = ff["Date"].astype(int)

months_needed = [202009, 202010, 202011, 202012, 202101]
ff_subset = ff[ff["Date"].isin(months_needed)].copy()

portfolio_cols = [col for col in ff_subset.columns if col != "Date"]
for col in portfolio_cols:
    ff_subset[col] = pd.to_numeric(ff_subset[col], errors="coerce")

# data matrix: rows = portfolios, columns = time periods
X = ff_subset.set_index("Date").T
print("Rows are the 25 portfolios; columns are Sep 2020 - Jan 2021.")
print("Shape of clustering data:", X.shape)

In [ ]:
# K-means clustering with K = 3
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=1, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

cluster_results = pd.DataFrame({"Portfolio": X.index, "Cluster": clusters})
print(cluster_results.sort_values("Cluster"))

for cluster_num in sorted(cluster_results["Cluster"].unique()):
    names = cluster_results[cluster_results["Cluster"] == cluster_num]["Portfolio"].tolist()
    print(f"\nCluster {cluster_num}:")
    for name in names:
        print(name)

X_with_clusters = X.copy()
X_with_clusters["Cluster"] = clusters
cluster_means = X_with_clusters.groupby("Cluster").mean()
print("\nAverage monthly returns by cluster:")
print(cluster_means)

**Result:** three clusters emerge with clear economic structure. Cluster 2 is exactly the
three smallest, most growth-oriented portfolios (`SMALL LoBM`, `ME1 BM2`, `ME2 BM1`) — small
size, low book-to-market. Cluster 0 groups higher book-to-market and large-cap value names
(`SMALL HiBM`, `BIG HiBM`) alongside a few mid-size ones. Cluster 1 is the large residual
group spanning most mid-size portfolios. The clusters recover a size/value structure similar
to the Fama-French sorting itself, purely from five months of realized co-movement — without
being told the size or book-to-market label of any portfolio.